<a href="https://colab.research.google.com/github/hilalozkaraman/leukemia-ml-llm-audit/blob/main/leukemia_ml_llm_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import re
import io
import json
import gzip
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import false_discovery_control  # scipy >= 1.11 (Benjamini-Hochberg)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score


In [12]:
# 1. SABİT RANDOM SEED VE GÖRSEL AYARLAR (IEEE Kalite Kıstası: >= 300 DPI veya vektör)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid", palette="deep")

OUTPUT_DIR = "./eda_outputs"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

RESULTS = {"golub": {}, "gse13159": {}, "cross_dataset": {}}  # rapordaki tüm sayılar buraya yazılır


In [13]:
# ==============================================================================
# ADIM 0: GOOGLE DRIVE BAĞLANTISI VE DOSYA YOLLARI
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

GOLUB_ZIP_PATH = "/content/drive/MyDrive/archive.zip"  # Kaggle Golub zip dosyası
GSE13159_GZ_PATH = "/content/drive/MyDrive/GSE13159_series_matrix.txt.gz"  # GEO dosyası
GPL570_PATH = "/content/drive/MyDrive/GPL570.annot.gz"  # (opsiyonel) ortak gen analizi için GEO anotasyonu
GOLUB_MAP_CSV = None  # (opsiyonel) 'probe,symbol' sütunlu CSV; Golub prob->gen sembolü eşlemesini ezer
EXTRACT_DIR = "./data_golub"

FLOOR, CEIL = 20.0, 16000.0  # Rehber 6.1: örnek alt-üst eşik
PAL_CLASS = {"ALL": "#2b83ba", "AML": "#d7191c"}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# ==============================================================================
# YARDIMCI FONKSİYONLAR
# ==============================================================================
def save_fig(name):
    """Şekli hem PNG (300 dpi) hem vektör PDF olarak kaydeder."""
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}.png"))
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}.pdf"))
    plt.close()


def save_clustermap(g, name):
    g.savefig(os.path.join(OUTPUT_DIR, f"{name}.png"))
    g.savefig(os.path.join(OUTPUT_DIR, f"{name}.pdf"))
    plt.close()


def clean_for_eda(X_raw, train_idx=None, is_log2=False, floor=FLOOR, ceil=CEIL):
    """BETİMSEL analiz için: INF->NaN, (eğitim) ortalaması ile doldur, eşik kırp, log2.
    Model hattında bu adımlar CV katı içinde, yalnızca eğitim verisiyle yeniden yapılmalıdır."""
    X = X_raw.replace([np.inf, -np.inf], np.nan)
    means = X.loc[train_idx].mean(axis=0) if train_idx is not None else X.mean(axis=0)
    X = X.fillna(means).fillna(0.0)  # ikinci fillna: tüm sütunu NaN olan genler için
    if is_log2:
        return X.astype(np.float32)
    return np.log2(X.clip(lower=floor, upper=ceil)).astype(np.float32)


def describe_values(X_raw):
    """Sonlu değerler üzerinden kalite özeti (NaN/INF varken min/maks 'nan' yazmasın diye)."""
    v = X_raw.values.astype(float)
    fin = v[np.isfinite(v)]
    return {
        "n_nan": int(np.isnan(v).sum()), "n_inf": int(np.isinf(v).sum()),
        "n_negative": int((fin < 0).sum()), "n_zero": int((fin == 0).sum()),
        "pct_negative_or_zero": float((fin <= 0).mean() * 100),
        "n_below_floor_20": int((fin < FLOOR).sum()), "n_above_ceil_16000": int((fin > CEIL).sum()),
        "min": float(fin.min()), "p01": float(np.percentile(fin, 1)), "median": float(np.median(fin)),
        "p99": float(np.percentile(fin, 99)), "max": float(fin.max()),
    }


def welch_de(X_log, y):
    """Welch t-testi (AML=1 vs ALL=0) + Benjamini-Hochberg FDR. BETİMSEL kullanım içindir."""
    a = X_log[y.values == 1].values.astype(float)
    b = X_log[y.values == 0].values.astype(float)
    t, p = stats.ttest_ind(a, b, equal_var=False, axis=0)
    t = np.where(np.isfinite(t), t, 0.0)
    p = np.where(np.isfinite(p), p, 1.0)
    return pd.DataFrame({"t_stat": t, "abs_t": np.abs(t), "p_value": p,
                         "fdr": false_discovery_control(p, method="bh"),
                         "log2FC_AML_vs_ALL": a.mean(axis=0) - b.mean(axis=0)}, index=X_log.columns)


def eta_squared(values, groups):
    """Bir PC skorunun bir kategorik değişkenle ilişkisi (varyansın kaç %'i gruplar arası)."""
    g = np.asarray(pd.Series(groups).astype(str))
    mu = values.mean()
    sst = float(((values - mu) ** 2).sum())
    if sst == 0:
        return float("nan")
    ssb = sum(len(values[g == k]) * (values[g == k].mean() - mu) ** 2 for k in np.unique(g))
    return float(ssb / sst)


def domain_auc(X, labels, n_pc=10):
    """İki grubun ifade profilinden ayırt edilebilirliği (5-kat CV AUC). ~0.5: fark yok; ~1.0: güçlü batch/platform
    etkisi. Denetimsiz PCA + lojistik regresyon; BETİMSEL bir teşhis ölçüsüdür."""
    Xs = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    k = int(min(n_pc, Xs.shape[0] - 1, Xs.shape[1]))
    P = PCA(n_components=k, random_state=RANDOM_SEED).fit_transform(Xs)
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)
    proba = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"),
                              P, np.asarray(labels), cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(labels, proba))


def stratified_sample(y, n_all=8, n_aml=7):
    """Boxplotlar için sınıfa göre dengeli rastgele örnek (v1'deki 'ilk 15 hasta' temsilî değildi)."""
    rng = np.random.default_rng(RANDOM_SEED)
    a = rng.choice(y[y == 0].index.values, min(n_all, (y == 0).sum()), replace=False)
    b = rng.choice(y[y == 1].index.values, min(n_aml, (y == 1).sum()), replace=False)
    return list(a) + list(b)


def pca_tsne_figure(X_log, y, group, name, title, top_var=None, perplexity=15):
    """PCA (scree dahil) + t-SNE şekilleri; PC skorlarını ve açıklanan varyansı döndürür."""
    X = X_log
    if top_var is not None and X.shape[1] > top_var:
        X = X[X.var(axis=0).nlargest(top_var).index]  # en yüksek varyanslı öznitelikler
    Xs = StandardScaler().fit_transform(X.values.astype(float))
    ncomp = int(min(10, Xs.shape[0] - 1, Xs.shape[1]))
    pca = PCA(n_components=ncomp, random_state=RANDOM_SEED)
    S = pca.fit_transform(Xs)
    evr = pca.explained_variance_ratio_
    v1, v2 = evr[0] * 100, evr[1] * 100
    P50 = PCA(n_components=int(min(50, Xs.shape[0] - 1, Xs.shape[1])), random_state=RANDOM_SEED).fit_transform(Xs)
    T = TSNE(n_components=2, perplexity=perplexity, random_state=RANDOM_SEED, init="pca").fit_transform(P50)

    class_names = y.map({0: "ALL", 1: "AML"}).values
    grp = np.asarray(group)
    legend_b = len(np.unique(grp)) <= 10
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    sns.scatterplot(x=S[:, 0], y=S[:, 1], hue=class_names, style=grp if legend_b else None,
                    palette=PAL_CLASS, s=45, alpha=0.85, ax=axes[0])
    axes[0].set_title(f"(a) PCA - Sınıfa Göre (PC1: %{v1:.1f}, PC2: %{v2:.1f})")
    axes[0].set_xlabel(f"PC1 (%{v1:.1f} Varyans)")
    axes[0].set_ylabel(f"PC2 (%{v2:.1f} Varyans)")
    sns.scatterplot(x=S[:, 0], y=S[:, 1], hue=grp, style=class_names, s=45, alpha=0.85,
                    palette="Dark2" if legend_b else "tab20", legend=legend_b, ax=axes[1])
    axes[1].set_title(f"(b) PCA - {title}")
    axes[1].set_xlabel(f"PC1 (%{v1:.1f} Varyans)")
    axes[1].set_ylabel(f"PC2 (%{v2:.1f} Varyans)")
    sns.scatterplot(x=T[:, 0], y=T[:, 1], hue=class_names, style=grp if legend_b else None,
                    palette=PAL_CLASS, s=45, alpha=0.85, ax=axes[2])
    axes[2].set_title(f"(c) t-SNE (Perplexity={perplexity})")
    axes[2].set_xlabel("t-SNE Boyut 1")
    axes[2].set_ylabel("t-SNE Boyut 2")
    save_fig(f"{name}_pca_tsne")

    fig, ax = plt.subplots(figsize=(5, 3.8))
    ax.bar(range(1, ncomp + 1), evr * 100, color="#4c78a8")
    ax.plot(range(1, ncomp + 1), np.cumsum(evr) * 100, "k.-")
    ax.set_xlabel("Bileşen")
    ax.set_ylabel("Açıklanan varyans (%)")
    ax.set_title("Scree + kümülatif varyans")
    save_fig(f"{name}_pca_scree")
    return S, evr


def heatmap_top30(X_log, y, de, name, title, label_func=None):
    """En ayırt edici 30 öznitelik için hiyerarşik kümelemeli ısı haritası (BETİMSEL)."""
    top = de.sort_values("abs_t", ascending=False).head(30).index
    Z = pd.DataFrame(StandardScaler().fit_transform(X_log[top].values.astype(float)),
                     index=X_log.index, columns=[label_func(p) if label_func else p for p in top]).T
    col_colors = y.map({0: PAL_CLASS["ALL"], 1: PAL_CLASS["AML"]})
    g = sns.clustermap(Z, cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5, col_colors=col_colors,
                       figsize=(13, 9), xticklabels=False, yticklabels=True)
    g.fig.suptitle(title, y=1.02)
    save_clustermap(g, name)


def load_gpl_annot(path):
    """GEO GPL anotasyon dosyasından (GPL570.annot[.gz]) prob -> gen sembolü / GenBank erişim tablosu."""
    opener = gzip.open if str(path).endswith(".gz") else open
    lines, started = [], False
    with opener(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                started = True
                continue
            if line.startswith("!platform_table_end"):
                break
            if started:
                lines.append(line)
    df = pd.read_csv(io.StringIO("".join(lines)), sep="\t", low_memory=False, dtype=str)
    low = {c.lower(): c for c in df.columns}
    sym_col = next(c for k, c in low.items() if "gene symbol" in k or k == "gene_symbol")
    gb_col = next((c for k, c in low.items() if "genbank" in k or k == "gb_acc"), None)
    out = pd.DataFrame({"ID": df[low["id"]],
                        "symbol": df[sym_col].fillna("").str.split("///").str[0].str.strip()})
    out["gb"] = df[gb_col].fillna("").str.split("///").str[0].str.strip() if gb_col else ""
    return out[(out["symbol"] != "") & (out["symbol"] != "---")]


def golub_symbol_map(golub_ids, gpl, override_csv=None):
    """Golub (Hu6800) prob -> gen sembolü. KISMİ kapsama: GenBank erişim numarası üzerinden GPL570 ile eşleştirir.
    Kapsama oranını raporla; iyileştirmek için GOLUB_MAP_CSV ('probe,symbol') ver."""
    if override_csv:
        m = pd.read_csv(override_csv)
        m.columns = [c.strip().lower() for c in m.columns]
        return m.set_index(m.columns[0])[m.columns[1]].astype(str)
    gb2sym = gpl[gpl["gb"] != ""].drop_duplicates("gb").set_index("gb")["symbol"]
    res = {}
    for gid in golub_ids:
        base = str(gid).split("#")[0]
        if base.startswith(("AFFX", "HG")):
            continue
        acc = base.split("_")[0]
        if acc in gb2sym.index:
            res[gid] = gb2sym[acc]
    return pd.Series(res, dtype=str)


def collapse_to_symbols(X_log, f2s):
    """Aynı gen sembolüne giden öznitelikler arasında ortalaması en yüksek olanı tutar."""
    s = f2s[f2s.index.isin(X_log.columns)].dropna()
    s = s[s.astype(str) != ""]
    if s.empty:
        return pd.DataFrame(index=X_log.index)
    d = pd.DataFrame({"sym": s.values, "mean": X_log[s.index].mean(axis=0).values}, index=s.index)
    d = d.sort_values("mean", ascending=False).drop_duplicates("sym")
    G = X_log[d.index].copy()
    G.columns = d["sym"].values
    return G
